# PHARVO-beta: POS Cart Quantity Change Test

**Objective:** Verify that an authorized staff user (`rafi`) can add a medicine (`Brufen`) to the **Current Sale** cart and increment its quantity from `1` to `3` using the stepper component, confirming accurate line item subtotal recalculation.

### Test Parameters
- **Medicine:** `Brufen`
- **Initial Quantity:** `1`
- **Target Quantity:** `3`
- **Operation:** Incrementing (`+`)

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Data ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password
MEDICINE_SEARCH = "Brufen"
INITIAL_QTY = 1
TARGET_QTY = 3

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print(f"[INFO] Starting Quantity Change Test for '{MEDICINE_SEARCH}' (Incrementing {INITIAL_QTY} -> {TARGET_QTY})...")

    # Step 2: Open login page and sign in
    driver.get(f"{BASE_URL}/")

    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 3: Navigate to POS / Sales module via sidebar
    pos_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'POS / Sales')]")
        )
    )
    pos_nav.click()

    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'POS / Sales')]")
        )
    )
    print("[INFO] Navigated to POS / Sales terminal.")

    # Step 4: Search for target medicine and add to cart
    pos_search_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Search medicine or brand' and contains(@class, 'pos-input')]")
        )
    )
    pos_search_input.clear()
    pos_search_input.send_keys(MEDICINE_SEARCH)

    target_row = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//table[contains(@class, 'pos-table')]//tbody//tr[contains(@class, 'pos-row-tr') and .//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{MEDICINE_SEARCH.lower()}')]]")
        )
    )

    add_btn = target_row.find_element(By.XPATH, ".//button[contains(., 'Add')]")
    add_btn.click()
    print(f"[INFO] Added '{MEDICINE_SEARCH}' to cart.")

    # Step 5: Locate the item row in Current Sale cart
    cart_item_row = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//tr[contains(@class, 'pos-row') and .//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{MEDICINE_SEARCH.lower()}')]]")
        )
    )

    # Verify initial quantity is 1
    qty_span = cart_item_row.find_element(By.XPATH, ".//span[contains(@class, 'pos-stepper-val')]")
    current_qty = int(qty_span.text.strip())
    print(f"[INFO] Initial cart quantity: {current_qty}")
    assert current_qty == INITIAL_QTY, f"Expected initial quantity {INITIAL_QTY}, got {current_qty}"

    # Step 6: Locate the increment (+) button in the PosStepper component
    # Accessible via aria-label="Increase quantity of ..." or second stepper button
    inc_button = cart_item_row.find_element(
        By.XPATH, ".//button[contains(@aria-label, 'Increase') or (contains(@class, 'pos-stepper-btn') and position()=2)]"
    )

    # Step 7: Increment quantity to reach TARGET_QTY (3)
    clicks_needed = TARGET_QTY - current_qty
    for i in range(clicks_needed):
        inc_button.click()
        time.sleep(0.3)  # Brief pause between clicks for state update
        print(f"[INFO] Clicked increment button (+). (Click {i + 1} of {clicks_needed})")

    # Step 8: Wait and verify updated quantity
    wait.until(lambda d: cart_item_row.find_element(By.XPATH, ".//span[contains(@class, 'pos-stepper-val')]").text.strip() == str(TARGET_QTY))
    final_qty = int(cart_item_row.find_element(By.XPATH, ".//span[contains(@class, 'pos-stepper-val')]").text.strip())

    # Step 9: Verify updated price total calculation
    unit_price_text = cart_item_row.find_element(By.XPATH, ".//td[3]").text
    total_price_text = cart_item_row.find_element(By.XPATH, ".//td[4]/div[1]").text

    unit_price_val = float(''.join(c for c in unit_price_text if c.isdigit() or c == '.'))
    total_price_val = float(''.join(c for c in total_price_text if c.isdigit() or c == '.'))

    expected_total = unit_price_val * TARGET_QTY

    # Step 10: Print PASS/FAIL information
    if final_qty == TARGET_QTY and abs(total_price_val - expected_total) < 0.01:
        print("PASS: Cart quantity increment test successful.")
        print(f"      - Final Quantity: {final_qty}")
        print(f"      - Unit Price: {unit_price_text}")
        print(f"      - Expected Line Total: ?{expected_total:.2f}")
        print(f"      - Actual Line Total: {total_price_text}")
    else:
        print(f"FAIL: Quantity mismatch or price discrepancy (Expected Qty {TARGET_QTY}, Got {final_qty}; Expected Total {expected_total}, Got {total_price_val}).")

except Exception as error:
    print(f"FAIL: Quantity Change test encountered error: {error}")

finally:
    # Step 11: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
